In [1]:
import pandas as pd

# Load the dataset
file_path = 'data_bike_sharing.csv'
df = pd.read_csv(file_path)

# Display the first few rows of the dataframe
df.head()


,dteday,hum,weathersit,holiday,season,atemp,temp,hr,casual,registered,cnt
0,2011-12-09,0.62,1,0,4,0.3485,0.36,16,24,226,250
1,2012-06-17,0.64,1,0,2,0.5152,0.54,4,2,16,18
2,2011-06-15,0.53,1,0,2,0.6212,0.62,23,17,90,107
3,2012-03-31,0.87,2,0,2,0.3485,0.36,8,19,126,145
4,2012-07-31,0.55,1,0,3,0.6970,0.76,18,99,758,857


In [5]:
Q1 = df['cnt'].quantile(0.25)
Q3 = df['cnt'].quantile(0.75)
IQR = Q3 - Q1

lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

df['cnt_capped'] = df['cnt'].apply(lambda x: lower_bound if x < lower_bound else (upper_bound if x > upper_bound else x))



In [6]:
Q1 = df['cnt'].quantile(0.25)
Q3 = df['cnt'].quantile(0.75)
IQR = Q3 - Q1

lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

# Capping the outliers
df['cnt_capped'] = df['cnt'].apply(lambda x: lower_bound if x < lower_bound else (upper_bound if x > upper_bound else x))

# Display the statistics before and after capping the outliers
before_stats = df['cnt'].describe()
after_stats = df['cnt_capped'].describe()

before_stats, after_stats


(count    12165.000000
 mean       189.271023
 std        181.223903
 min          1.000000
 25%         40.000000
 50%        142.000000
 75%        282.000000
 max        970.000000
 Name: cnt, dtype: float64,
 count    12165.000000
 mean       186.320099
 std        172.190064
 min          1.000000
 25%         40.000000
 50%        142.000000
 75%        282.000000
 max        645.000000
 Name: cnt_capped, dtype: float64)

In [12]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

# Check for missing values
df.isnull().sum()

# Define features and target variable
X = df.drop(['cnt', 'cnt_capped'], axis=1)
y = df['cnt_capped']

# Identify numerical and categorical features
numerical_features = ['temp', 'atemp', 'hum', 'hr']
categorical_features = ['season', 'weathersit', 'holiday']

# Preprocess the data
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numerical_features),
        ('cat', OneHotEncoder(), categorical_features)
    ],
    remainder='passthrough'
)

# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)


In [13]:
from sklearn.preprocessing import PolynomialFeatures

# Create polynomial features
poly = PolynomialFeatures(degree=2, interaction_only=True, include_bias=False)

# Transform the features
X_train_poly = poly.fit_transform(preprocessor.fit_transform(X_train))
X_test_poly = poly.transform(preprocessor.transform(X_test))


ValueError: could not convert string to float: '2012-08-25'